# From a forecast to executable P&L
A signal predicts a small midpoint move. Work through a hypothetical round trip and test how its costs change the apparent edge. Run top to bottom in a Python 3 notebook.

## 1 · Cross the spread twice
The initial midpoint is 50 dollars and the full spread is 6 cents. Buy at the ask. Later the midpoint rises 8 cents and you sell at the bid with the same spread. Each fill costs half a cent per share. Predict the net profit on 100 shares.

In [1]:
mid = 50
spread = 0.06
move = 0.08
fee = 0.005
quantity = 100
buy = mid + spread / 2
sell = mid + move - spread / 2
net = quantity * (sell - buy - 2 * fee)
print(f"Buy {buy:.2f}, sell {sell:.2f}, net ${net:.2f}")

Buy 50.03, sell 50.05, net $1.00


**Worked check.** The midpoint move earns 8 cents per share before costs. Two half-spreads consume 6 cents, and the two fees consume another cent. One cent per share survives. Increase the spread to 8 cents and predict the sign before rerunning.

## 2 · Set the execution clock
Each signal below is computed *after* its closing price is final. Assume a signal of 1 buys at the next open and sells at that day's close, and a signal of 0 stays flat. For this timing exercise, use the listed prices and ignore costs. Which signal can earn Tuesday's open-to-close move?

In [2]:
days = ["Mon", "Tue", "Wed", "Thu"]
opens = [100, 103, 105, 101]
closes = [102, 105, 101, 102]
signals_after_close = [1, 0, 1, 0]
total = 0
for i in range(1, len(days)):
    position = signals_after_close[i - 1]
    pnl = position * (closes[i] - opens[i])
    total += pnl
    print(f"{days[i]} | prior signal {position} | gross P&L per share {pnl:+.2f}")
print(f"Total per share {total:+.2f}")

Tue | prior signal 1 | gross P&L per share +2.00
Wed | prior signal 0 | gross P&L per share +0.00
Thu | prior signal 1 | gross P&L per share +1.00
Total per share +3.00


**Worked check.** Monday's signal earns Tuesday's 2 dollars from open to close. Wednesday's signal earns Thursday's 1 dollar. Neither position earns an overnight gap. Flip Tuesday's signal and rerun to see which day's profit changes.

## 3 · Find the break-even spread
Return to the first trade. Change the forecast move, run the cell, and explain why doubling the order quantity leaves the per-share break-even spread unchanged in this model.

In [3]:
forecast_move = 0.08
break_even_spread = forecast_move - 2 * fee
print(f"Break-even full spread: {100 * break_even_spread:.1f} cents")
for trial_spread in [0.04, 0.07, 0.10]:
    print(f"Spread {trial_spread:.2f} | net ${quantity * (forecast_move - trial_spread - 2 * fee):.2f}")

Break-even full spread: 7.0 cents
Spread 0.04 | net $3.00
Spread 0.07 | net $-0.00
Spread 0.10 | net $-3.00


**Interview follow-up.** Which assumption would you challenge first for a much larger order?

**Worked check.** The model fills the entire order at the best quote. Compare order size to available depth, then model the volume-weighted execution price across levels. Larger orders can also move future quotes.